In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

print("="*80)
print("NATURAL GAS STORAGE CONTRACT PRICING MODEL")
print("JP Morgan Quantitative Research - Task 2")
print("="*80)

# ============================================================================
# PART 1: LOAD DATA AND BUILD PRICE ESTIMATION MODEL (FROM TASK 1)
# ============================================================================

print("\n[STEP 1] Loading historical data and building price model...")

# Load the data
df = pd.read_csv('Nat_Gas.csv')
df['Dates'] = pd.to_datetime(df['Dates'], format='%m/%d/%y')
df = df.sort_values('Dates').reset_index(drop=True)

print(f"Data loaded: {len(df)} records from {df['Dates'].min().date()} to {df['Dates'].max().date()}")

# Feature engineering
df['Month'] = df['Dates'].dt.month
df['Days_Since_Start'] = (df['Dates'] - df['Dates'].min()).dt.days
df['Sin_Month'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Cos_Month'] = np.cos(2 * np.pi * df['Month'] / 12)

# Prepare features for modeling
X = df[['Days_Since_Start', 'Sin_Month', 'Cos_Month']].values
y = df['Prices'].values

# Use polynomial features to capture non-linear trends
poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(X)

# Train the model
model = LinearRegression()
model.fit(X_poly, y)

print("✓ Price estimation model trained successfully")

# ============================================================================
# PRICE ESTIMATION FUNCTION (FROM TASK 1)
# ============================================================================

def estimate_gas_price(input_date):
    """
    Estimate natural gas price for any given date.
    
    Parameters:
    -----------
    input_date : str or datetime
        Date for price estimation (format: 'YYYY-MM-DD' or datetime object)
    
    Returns:
    --------
    float
        Estimated price in $/MMBtu
    """
    # Convert string to datetime
    if isinstance(input_date, str):
        date = pd.to_datetime(input_date)
    else:
        date = input_date
    
    # Extract features
    month = date.month
    days_since_start = (date - df['Dates'].min()).days
    sin_month = np.sin(2 * np.pi * month / 12)
    cos_month = np.cos(2 * np.pi * month / 12)
    
    # Prepare input and predict
    X_input = np.array([[days_since_start, sin_month, cos_month]])
    X_input_poly = poly.transform(X_input)
    price = model.predict(X_input_poly)[0]
    
    return price

# ============================================================================
# PART 2: CONTRACT PRICING MODEL (TASK 2)
# ============================================================================

print("\n[STEP 2] Building contract pricing model...")

def price_contract(
    injection_dates,
    withdrawal_dates,
    injection_rates,
    withdrawal_rates,
    max_storage_volume,
    storage_cost_per_month
):
    """
    Price a natural gas storage contract based on injection/withdrawal schedules.
    
    This function calculates the net present value of a storage contract by:
    1. Calculating purchase costs (injection dates × volumes × prices)
    2. Calculating sale revenues (withdrawal dates × volumes × prices)
    3. Deducting storage costs based on the storage duration
    4. Deducting injection/withdrawal fees
    
    Parameters:
    -----------
    injection_dates : list of str or datetime
        Dates when gas will be injected (purchased and stored)
        Format: ['YYYY-MM-DD', 'YYYY-MM-DD', ...]
        
    withdrawal_dates : list of str or datetime
        Dates when gas will be withdrawn (sold)
        Format: ['YYYY-MM-DD', 'YYYY-MM-DD', ...]
        
    injection_rates : list of tuples
        Volume and cost for each injection: [(volume_MMBtu, cost_per_MMBtu), ...]
        - volume_MMBtu: Amount of gas to inject on that date
        - cost_per_MMBtu: Fee charged by storage facility per MMBtu injected
        
    withdrawal_rates : list of tuples
        Volume and cost for each withdrawal: [(volume_MMBtu, cost_per_MMBtu), ...]
        - volume_MMBtu: Amount of gas to withdraw on that date
        - cost_per_MMBtu: Fee charged by storage facility per MMBtu withdrawn
        
    max_storage_volume : float
        Maximum storage capacity in MMBtu
        
    storage_cost_per_month : float
        Fixed monthly storage fee in dollars
    
    Returns:
    --------
    dict
        Dictionary containing:
        - 'contract_value': Net value of the contract in dollars
        - 'total_revenue': Total revenue from gas sales
        - 'total_purchase_cost': Total cost of purchasing gas
        - 'total_injection_cost': Total injection fees
        - 'total_withdrawal_cost': Total withdrawal fees
        - 'total_storage_cost': Total storage fees
        - 'volume_profile': DataFrame showing storage levels over time
        - 'cash_flows': DataFrame showing all cash flows
    
    Raises:
    -------
    ValueError
        If storage capacity is exceeded or withdrawal exceeds available volume
    
    Examples:
    ---------
    >>> # Simple contract: inject in summer, withdraw in winter
    >>> result = price_contract(
    ...     injection_dates=['2024-07-01'],
    ...     withdrawal_dates=['2024-12-01'],
    ...     injection_rates=[(1000000, 0.01)],  # 1M MMBtu at $0.01/MMBtu fee
    ...     withdrawal_rates=[(1000000, 0.01)],
    ...     max_storage_volume=2000000,
    ...     storage_cost_per_month=100000
    ... )
    >>> print(f"Contract Value: ${result['contract_value']:,.2f}")
    """
    
    # Convert dates to datetime objects
    injection_dates = [pd.to_datetime(d) for d in injection_dates]
    withdrawal_dates = [pd.to_datetime(d) for d in withdrawal_dates]
    
    # Validate inputs
    if len(injection_dates) != len(injection_rates):
        raise ValueError("Number of injection dates must match number of injection rates")
    if len(withdrawal_dates) != len(withdrawal_rates):
        raise ValueError("Number of withdrawal dates must match number of withdrawal rates")
    
    # Initialize tracking variables
    cash_flows = []
    current_volume = 0
    max_volume_reached = 0
    
    # ========================================================================
    # STEP 1: Process all injections (purchases)
    # ========================================================================
    
    total_purchase_cost = 0
    total_injection_cost = 0
    
    for date, (volume, injection_fee_per_mmbtu) in zip(injection_dates, injection_rates):
        # Check storage capacity
        if current_volume + volume > max_storage_volume:
            raise ValueError(
                f"Storage capacity exceeded on {date.date()}: "
                f"Attempting to store {current_volume + volume:,.0f} MMBtu "
                f"but max capacity is {max_storage_volume:,.0f} MMBtu"
            )
        
        # Get market price for this date
        market_price = estimate_gas_price(date)
        
        # Calculate costs
        purchase_cost = volume * market_price
        injection_fee = volume * injection_fee_per_mmbtu
        
        # Update totals
        total_purchase_cost += purchase_cost
        total_injection_cost += injection_fee
        current_volume += volume
        max_volume_reached = max(max_volume_reached, current_volume)
        
        # Record cash flow (outflow is negative)
        cash_flows.append({
            'date': date,
            'type': 'Injection',
            'volume': volume,
            'price_per_mmbtu': market_price,
            'market_value': -purchase_cost,
            'fee': -injection_fee,
            'net_cash_flow': -(purchase_cost + injection_fee),
            'storage_level': current_volume
        })
    
    # ========================================================================
    # STEP 2: Process all withdrawals (sales)
    # ========================================================================
    
    total_revenue = 0
    total_withdrawal_cost = 0
    
    for date, (volume, withdrawal_fee_per_mmbtu) in zip(withdrawal_dates, withdrawal_rates):
        # Check available volume
        if volume > current_volume:
            raise ValueError(
                f"Insufficient gas on {date.date()}: "
                f"Attempting to withdraw {volume:,.0f} MMBtu "
                f"but only {current_volume:,.0f} MMBtu available"
            )
        
        # Get market price for this date
        market_price = estimate_gas_price(date)
        
        # Calculate revenues and costs
        sale_revenue = volume * market_price
        withdrawal_fee = volume * withdrawal_fee_per_mmbtu
        
        # Update totals
        total_revenue += sale_revenue
        total_withdrawal_cost += withdrawal_fee
        current_volume -= volume
        
        # Record cash flow (inflow is positive, fees are negative)
        cash_flows.append({
            'date': date,
            'type': 'Withdrawal',
            'volume': -volume,  # Negative to show outflow
            'price_per_mmbtu': market_price,
            'market_value': sale_revenue,
            'fee': -withdrawal_fee,
            'net_cash_flow': sale_revenue - withdrawal_fee,
            'storage_level': current_volume
        })
    
    # ========================================================================
    # STEP 3: Calculate storage costs
    # ========================================================================
    
    # Find the storage period (from first injection to last withdrawal)
    if injection_dates and withdrawal_dates:
        start_date = min(injection_dates)
        end_date = max(withdrawal_dates)
        
        # Calculate number of months (rounding up for partial months)
        days_stored = (end_date - start_date).days
        months_stored = np.ceil(days_stored / 30)  # Approximate months
        
        total_storage_cost = months_stored * storage_cost_per_month
    else:
        months_stored = 0
        total_storage_cost = 0
    
    # ========================================================================
    # STEP 4: Calculate net contract value
    # ========================================================================
    
    contract_value = (
        total_revenue 
        - total_purchase_cost 
        - total_injection_cost 
        - total_withdrawal_cost 
        - total_storage_cost
    )
    
    # ========================================================================
    # STEP 5: Prepare detailed output
    # ========================================================================
    
    # Create DataFrame of cash flows
    cash_flows_df = pd.DataFrame(cash_flows)
    cash_flows_df = cash_flows_df.sort_values('date').reset_index(drop=True)
    
    # Create summary dictionary
    result = {
        'contract_value': contract_value,
        'total_revenue': total_revenue,
        'total_purchase_cost': total_purchase_cost,
        'total_injection_cost': total_injection_cost,
        'total_withdrawal_cost': total_withdrawal_cost,
        'total_storage_cost': total_storage_cost,
        'storage_duration_months': months_stored,
        'max_storage_used': max_volume_reached,
        'cash_flows': cash_flows_df
    }
    
    return result


def print_contract_summary(result):
    """
    Print a formatted summary of the contract valuation.
    
    Parameters:
    -----------
    result : dict
        Output from price_contract() function
    """
    print("\n" + "="*80)
    print("CONTRACT VALUATION SUMMARY")
    print("="*80)
    
    print("\n CASH FLOW BREAKDOWN:")
    print(f"  Revenue from Sales:        ${result['total_revenue']:>15,.2f}")
    print(f"  Cost of Purchases:        -${result['total_purchase_cost']:>15,.2f}")
    print(f"  Injection Fees:           -${result['total_injection_cost']:>15,.2f}")
    print(f"  Withdrawal Fees:          -${result['total_withdrawal_cost']:>15,.2f}")
    print(f"  Storage Costs ({result['storage_duration_months']:.0f} months): -${result['total_storage_cost']:>15,.2f}")
    print("  " + "-"*76)
    print(f"  NET CONTRACT VALUE:        ${result['contract_value']:>15,.2f}")
    
    print("\n OPERATIONAL DETAILS:")
    print(f"  Storage Duration:          {result['storage_duration_months']:.1f} months")
    print(f"  Max Storage Used:          {result['max_storage_used']:,.0f} MMBtu")
    
    print("\n DETAILED CASH FLOWS:")
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
    print(result['cash_flows'].to_string(index=False))
    
    print("\n" + "="*80)


# ============================================================================
# PART 3: TEST CASES
# ============================================================================

print("\n[STEP 3] Running test cases...\n")

# ----------------------------------------------------------------------------
# TEST CASE 1: Simple Summer-to-Winter Trade
# ----------------------------------------------------------------------------

print("\n" + "="*80)
print("TEST CASE 1: Simple Summer-to-Winter Storage")
print("="*80)
print("\nScenario:")
print("  • Inject 1,000,000 MMBtu in July 2024 (summer - low prices)")
print("  • Withdraw 1,000,000 MMBtu in January 2025 (winter - high prices)")
print("  • Storage capacity: 2,000,000 MMBtu")
print("  • Storage cost: $100,000/month")
print("  • Injection fee: $0.01/MMBtu")
print("  • Withdrawal fee: $0.01/MMBtu")

result1 = price_contract(
    injection_dates=['2024-07-01'],
    withdrawal_dates=['2025-01-15'],
    injection_rates=[(1_000_000, 0.01)],
    withdrawal_rates=[(1_000_000, 0.01)],
    max_storage_volume=2_000_000,
    storage_cost_per_month=100_000
)

print_contract_summary(result1)

# ----------------------------------------------------------------------------
# TEST CASE 2: Multiple Injections and Withdrawals
# ----------------------------------------------------------------------------

print("\n" + "="*80)
print("TEST CASE 2: Multiple Injection/Withdrawal Schedule")
print("="*80)
print("\nScenario:")
print("  • Inject in two batches: June and July 2024")
print("  • Withdraw in two batches: December 2024 and January 2025")
print("  • Storage capacity: 3,000,000 MMBtu")
print("  • Storage cost: $150,000/month")

result2 = price_contract(
    injection_dates=['2024-06-01', '2024-07-15'],
    withdrawal_dates=['2024-12-15', '2025-01-31'],
    injection_rates=[
        (800_000, 0.01),   # First injection: 800K MMBtu
        (700_000, 0.01)    # Second injection: 700K MMBtu
    ],
    withdrawal_rates=[
        (900_000, 0.01),   # First withdrawal: 900K MMBtu
        (600_000, 0.01)    # Second withdrawal: 600K MMBtu
    ],
    max_storage_volume=3_000_000,
    storage_cost_per_month=150_000
)

print_contract_summary(result2)

# ----------------------------------------------------------------------------
# TEST CASE 3: High Volume, Long Duration
# ----------------------------------------------------------------------------

print("\n" + "="*80)
print("TEST CASE 3: Large Volume, Extended Storage Period")
print("="*80)
print("\nScenario:")
print("  • Inject 2,500,000 MMBtu in May 2024")
print("  • Withdraw 2,500,000 MMBtu in February 2025")
print("  • Storage capacity: 5,000,000 MMBtu")
print("  • Storage cost: $200,000/month")
print("  • Higher injection/withdrawal fees: $0.015/MMBtu")

result3 = price_contract(
    injection_dates=['2024-05-15'],
    withdrawal_dates=['2025-02-28'],
    injection_rates=[(2_500_000, 0.015)],
    withdrawal_rates=[(2_500_000, 0.015)],
    max_storage_volume=5_000_000,
    storage_cost_per_month=200_000
)

print_contract_summary(result3)

# ----------------------------------------------------------------------------
# TEST CASE 4: Testing Edge Cases
# ----------------------------------------------------------------------------

print("\n" + "="*80)
print("TEST CASE 4: Short-Term Storage (Summer to Fall)")
print("="*80)
print("\nScenario:")
print("  • Inject in August 2024")
print("  • Withdraw in October 2024 (shorter hold period)")
print("  • Testing if strategy works for non-winter withdrawal")

result4 = price_contract(
    injection_dates=['2024-08-01'],
    withdrawal_dates=['2024-10-15'],
    injection_rates=[(500_000, 0.01)],
    withdrawal_rates=[(500_000, 0.01)],
    max_storage_volume=1_000_000,
    storage_cost_per_month=50_000
)

print_contract_summary(result4)

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("PRICING MODEL VALIDATION COMPLETE")
print("="*80)

print("\n MODEL CAPABILITIES:")
print("  • Handles multiple injection dates")
print("  • Handles multiple withdrawal dates")
print("  • Validates storage capacity constraints")
print("  • Validates volume availability for withdrawals")
print("  • Calculates all relevant costs (purchase, injection, withdrawal, storage)")
print("  • Provides detailed cash flow analysis")
print("  • Uses Task 1 price estimation for market prices")

print("\n TEST RESULTS SUMMARY:")
print(f"  Test Case 1 (Simple):        ${result1['contract_value']:>12,.2f}")
print(f"  Test Case 2 (Multiple):      ${result2['contract_value']:>12,.2f}")
print(f"  Test Case 3 (Large Volume):  ${result3['contract_value']:>12,.2f}")
print(f"  Test Case 4 (Short-Term):    ${result4['contract_value']:>12,.2f}")

print("\n READY FOR PRODUCTION:")
print("  The model is ready for:")
print("  • Manual oversight by trading desk")
print("  • Client option exploration")
print("  • Further validation and testing")
print("  • Integration with engineering and risk systems")

print("\n" + "="*80)
print("You can now use the price_contract() function to value any storage contract!")
print("="*80)

NATURAL GAS STORAGE CONTRACT PRICING MODEL
JP Morgan Quantitative Research - Task 2

[STEP 1] Loading historical data and building price model...
Data loaded: 48 records from 2020-10-31 to 2024-09-30
✓ Price estimation model trained successfully

[STEP 2] Building contract pricing model...

[STEP 3] Running test cases...


TEST CASE 1: Simple Summer-to-Winter Storage

Scenario:
  • Inject 1,000,000 MMBtu in July 2024 (summer - low prices)
  • Withdraw 1,000,000 MMBtu in January 2025 (winter - high prices)
  • Storage capacity: 2,000,000 MMBtu
  • Storage cost: $100,000/month
  • Injection fee: $0.01/MMBtu
  • Withdrawal fee: $0.01/MMBtu

CONTRACT VALUATION SUMMARY

 CASH FLOW BREAKDOWN:
  Revenue from Sales:        $  13,271,628.85
  Cost of Purchases:        -$  11,490,839.16
  Injection Fees:           -$      10,000.00
  Withdrawal Fees:          -$      10,000.00
  Storage Costs (7 months): -$     700,000.00
  ------------------------------------------------------------------------